# Student Performance — Notebook 5: Advanced Additions
**Project:** End-to-End AI Framework for Student Performance Analysis
**Student:** Megha Deopa | PRN: 2405020011520 | MBA AI/ML July 2024

**What is in this notebook:**
1. K-Means Clustering — Student Segmentation (High / Average / At-Risk)
2. Scenario / What-If Analysis — Prescriptive Analytics
3. Rule-Based Recommendation System

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline
import warnings
warnings.filterwarnings('ignore')

from sklearn.cluster         import KMeans
from sklearn.decomposition   import PCA
from sklearn.preprocessing   import OneHotEncoder, StandardScaler
from sklearn.compose         import ColumnTransformer
from sklearn.linear_model    import LinearRegression
from sklearn.model_selection import train_test_split

# Load data
df = pd.read_csv('data/stud.csv')
df['total score'] = df['math_score'] + df['reading_score'] + df['writing_score']
df['average']     = df['total score'] / 3
df['pass_fail']   = df['average'].apply(lambda x: 'Pass' if x >= 40 else 'Fail')

def assign_grade(score):
    if   score >= 90: return 'A'
    elif score >= 75: return 'B'
    elif score >= 60: return 'C'
    elif score >= 40: return 'D'
    else:             return 'F'

df['grade'] = df['average'].apply(assign_grade)
print('Dataset ready! Shape:', df.shape)

## PART 1: K-Means Clustering — Student Segmentation

**What is K-Means?** An unsupervised ML algorithm that groups students by similarity of scores WITHOUT being told the answer.

**Why is this important?** It lets us discover natural student performance groups from data alone — no labels needed.

### Step 1: Scale the Score Columns

In [ ]:
# Use only 3 score columns for clustering
score_cols    = df[['math_score','reading_score','writing_score']]
scaler        = StandardScaler()
scores_scaled = scaler.fit_transform(score_cols)
print('Scores scaled. Shape:', scores_scaled.shape)

### Step 2: Elbow Method — Find Optimal K

In [ ]:
# Calculate WCSS for K = 1 to 10
wcss = []
for k in range(1, 11):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(scores_scaled)
    wcss.append(km.inertia_)

plt.figure(figsize=(10, 5))
plt.plot(range(1,11), wcss, marker='o', color='#3498db', linewidth=2, markersize=8)
plt.fill_between(range(1,11), wcss, alpha=0.1, color='#3498db')
plt.axvline(x=3, color='red', linestyle='--', linewidth=1.5, label='Optimal K = 3')
plt.title('Elbow Method — Finding Optimal Number of Clusters', fontsize=14)
plt.xlabel('Number of Clusters (K)')
plt.ylabel('WCSS (Within Cluster Sum of Squares)')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()
print('Optimal K = 3 (High Performer, Average Performer, At-Risk)')

### Step 3: Apply K-Means with K=3

In [ ]:
km_final = KMeans(n_clusters=3, random_state=42, n_init=10)
df['cluster'] = km_final.fit_predict(scores_scaled)

# Label clusters based on average score
cluster_means = df.groupby('cluster')['average'].mean().sort_values()
label_map = {
    cluster_means.index[0]: 'At-Risk',
    cluster_means.index[1]: 'Average Performer',
    cluster_means.index[2]: 'High Performer'
}
df['segment'] = df['cluster'].map(label_map)

print('Cluster sizes:')
print(df['segment'].value_counts())

### Step 4: Visualise Clusters using PCA

In [ ]:
# PCA reduces 3 dimensions → 2 for easy plotting
pca = PCA(n_components=2)
scores_pca    = pca.fit_transform(scores_scaled)
df['pca_1']   = scores_pca[:, 0]
df['pca_2']   = scores_pca[:, 1]

plt.figure(figsize=(11, 7))
colors = {'High Performer':'#27ae60','Average Performer':'#f39c12','At-Risk':'#e74c3c'}

for segment, color in colors.items():
    mask = df['segment'] == segment
    plt.scatter(df[mask]['pca_1'], df[mask]['pca_2'],
                c=color, label=segment, alpha=0.7, s=60,
                edgecolors='white', linewidth=0.5)

plt.title('Student Segmentation — K-Means Clustering (K=3)\nVisualised using PCA', fontsize=14)
plt.xlabel(f'PC1 (explains {pca.explained_variance_ratio_[0]*100:.1f}% variance)')
plt.ylabel(f'PC2 (explains {pca.explained_variance_ratio_[1]*100:.1f}% variance)')
plt.legend(title='Student Segment', fontsize=11)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f'PCA captures {pca.explained_variance_ratio_.sum()*100:.1f}% of total variance')

### Step 5: Cluster Profile Analysis

In [ ]:
# Average scores per cluster
profile = df.groupby('segment')[['math_score','reading_score','writing_score','average']].mean().round(2)
print('=== Cluster Profile (Average Scores) ===')
print(profile)

print('\n=== Lunch Distribution by Segment ===')
print(pd.crosstab(df['segment'], df['lunch']))

print('\n=== Test Prep by Segment ===')
print(pd.crosstab(df['segment'], df['test preparation course']))

# Bar chart of segment sizes
plt.figure(figsize=(10, 5))
seg_counts = df['segment'].value_counts()
colors_bar = ['#27ae60','#f39c12','#e74c3c']
bars = plt.bar(seg_counts.index, seg_counts.values, color=colors_bar, width=0.4, alpha=0.85)
for bar in bars:
    plt.text(bar.get_x() + bar.get_width()/2,
             bar.get_height() + 5,
             str(int(bar.get_height())),
             ha='center', fontsize=13, fontweight='bold')
plt.title('Number of Students in Each Performance Segment', fontsize=14)
plt.ylabel('Number of Students')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## PART 2: Scenario / What-If Analysis

**What is this?** Using the trained model to predict how a student's score CHANGES when we change one factor.

**Why is this important?** This is PRESCRIPTIVE ANALYTICS — not just predicting what will happen, but what institutions SHOULD DO to improve outcomes.

**3 Levels of Analytics:**
- Descriptive → What happened (EDA)
- Predictive  → What will happen (regression/classification)
- Prescriptive → What should we do (scenario analysis — this notebook)

In [ ]:
# Train the prediction model first
X = df.drop(columns=['math_score','total score','average','pass_fail','grade',
                      'cluster','segment','pca_1','pca_2'])
y = df['math_score']

cat_f = X.select_dtypes(include='object').columns
num_f = X.select_dtypes(exclude='object').columns

pre = ColumnTransformer([
    ('ohe',    OneHotEncoder(handle_unknown='ignore'), cat_f),
    ('scaler', StandardScaler(),                       num_f)
])

Xp = pre.fit_transform(X)
model = LinearRegression()
model.fit(Xp, y)
print('Model trained! R² on full data:', round(model.score(Xp, y), 4))

In [ ]:
def predict_score(gender, race, parent_edu, lunch, test_prep, reading, writing):
    """Predict math score from student profile"""
    inp  = pd.DataFrame([[gender, race, parent_edu, lunch, test_prep, reading, writing]],
                         columns=X.columns)
    pred = model.predict(pre.transform(inp))[0]
    return round(float(np.clip(pred, 0, 100)), 2)

print('Prediction function ready!')

In [ ]:
# ---- SCENARIO 1: Impact of Test Preparation ----
print('=' * 60)
print('SCENARIO 1 — Impact of Test Preparation Course')
print('=' * 60)
before1 = predict_score('female','group C','some college','free/reduced','none', 65, 68)
after1  = predict_score('female','group C','some college','free/reduced','completed', 65, 68)
print(f'WITHOUT test prep: {before1}')
print(f'WITH test prep:    {after1}')
print(f'Improvement:       +{after1 - before1:.2f} points')

# ---- SCENARIO 2: Impact of Lunch Type ----
print('\n' + '=' * 60)
print('SCENARIO 2 — Impact of Lunch Type')
print('=' * 60)
before2 = predict_score('male','group B','high school','free/reduced','none', 60, 62)
after2  = predict_score('male','group B','high school','standard',     'none', 60, 62)
print(f'FREE/REDUCED lunch: {before2}')
print(f'STANDARD lunch:     {after2}')
print(f'Improvement:        +{after2 - before2:.2f} points')

# ---- SCENARIO 3: Both Changes Combined ----
print('\n' + '=' * 60)
print('SCENARIO 3 — Both Interventions Combined')
print('=' * 60)
base     = predict_score('female','group A','high school','free/reduced','none', 55, 58)
improved = predict_score('female','group A','high school','standard','completed', 55, 58)
print(f'Baseline (no changes):       {base}')
print(f'After both improvements:     {improved}')
print(f'Total potential improvement: +{improved - base:.2f} points')

In [ ]:
# Visualise all 3 scenarios
scenarios     = ['Test Prep\n(Scenario 1)','Lunch Type\n(Scenario 2)','Both Combined\n(Scenario 3)']
before_scores = [before1, before2, base]
after_scores  = [after1,  after2,  improved]

x     = np.arange(len(scenarios))
width = 0.3

fig, ax = plt.subplots(figsize=(12, 6))
b1 = ax.bar(x - width/2, before_scores, width, label='Before', color='#e74c3c', alpha=0.85)
b2 = ax.bar(x + width/2, after_scores,  width, label='After',  color='#27ae60', alpha=0.85)

for bar in list(b1) + list(b2):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.5,
            f'{bar.get_height():.1f}',
            ha='center', va='bottom', fontsize=11, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(scenarios, fontsize=11)
ax.set_ylabel('Predicted Math Score')
ax.set_title('Scenario Analysis — Predicted Score Before vs After Intervention', fontsize=14)
ax.legend(fontsize=11)
ax.set_ylim(0, 90)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## PART 3: Rule-Based Recommendation System

**What is this?** A system that looks at a student's profile and gives PERSONALISED suggestions to improve their score.

**This makes your project PRESCRIPTIVE** — it doesn't just predict, it recommends what to do.

In [ ]:
def get_recommendations(gender, race, parent_edu, lunch, test_prep, reading, writing, predicted_math):
    """Generate personalised recommendations based on student profile"""
    recommendations = []

    if test_prep == 'none':
        recommendations.append('[HIGH] Enrol in the Test Preparation Course — improves scores by 5-8 points')

    if lunch == 'free/reduced':
        recommendations.append('[HIGH] Apply for Standard Meal Program — standard lunch students score 8-10 pts higher')

    if reading < 60:
        recommendations.append('[MEDIUM] Join Reading Improvement Program — reading has 0.82 correlation with math')

    if writing < 60:
        recommendations.append('[MEDIUM] Attend Writing Support Sessions — writing and reading reinforce each other')

    if race in ['group a', 'group b', 'group A', 'group B']:
        recommendations.append('[MEDIUM] Access Targeted Academic Support — Groups A and B benefit most from mentoring')

    if parent_edu in ['some high school', 'high school']:
        recommendations.append('[LOW] Encourage Parent Engagement Programs — parental education positively impacts scores')

    if predicted_math >= 75:
        recommendations.append('[INFO] Strong performer — consider advanced coursework or scholarship opportunities')
    elif predicted_math >= 50:
        recommendations.append('[INFO] Average performer — consistent study + test prep will push to higher grade')
    else:
        recommendations.append('[URGENT] At-Risk student — immediate academic counselling strongly advised')

    return recommendations

In [ ]:
# Test on 3 different student profiles
profiles = [
    {'name':'Student A (At-Risk)',       'gender':'male',  'race':'group A','parent_edu':'high school',
     'lunch':'free/reduced','test_prep':'none','reading':48,'writing':50},
    {'name':'Student B (Average)',       'gender':'female','race':'group C','parent_edu':'some college',
     'lunch':'standard',    'test_prep':'none','reading':68,'writing':70},
    {'name':'Student C (High Performer)','gender':'female','race':'group E','parent_edu':"master's degree",
     'lunch':'standard',    'test_prep':'completed','reading':88,'writing':90}
]

for p in profiles:
    pred = predict_score(p['gender'], p['race'], p['parent_edu'],
                          p['lunch'], p['test_prep'], p['reading'], p['writing'])
    recs = get_recommendations(p['gender'], p['race'], p['parent_edu'],
                                p['lunch'], p['test_prep'], p['reading'], p['writing'], pred)
    print('=' * 65)
    print(f"{p['name']}")
    print(f'Predicted Math Score: {pred}')
    print('Personalised Recommendations:')
    for rec in recs:
        print(f'  • {rec}')
    print()